In [ ]:
import pandas as pd
import numpy as np
import sympy as sp
import math
import scipy
import matplotlib.pyplot as plt
import scikit_posthocs as sp
import statsmodels.api as sm
from statsmodels.stats.anova import AnovaRM
import pingouin as pg

In [ ]:
df = pd.read_csv('combine_formal.csv')
display(df)

# Major Comparison

In [ ]:
stage_list = ['1 puzzle', '2 shooting', '3 maze', '4 city']

for i, k in enumerate(df.columns):
  if (i < 3): continue
  print("\n┉┉┉┉┉┉┉┉┉┉┉┉\n🔷Column", k)
  for j, l in enumerate(stage_list):
    print("\n🔹Stage", l)
    none = df[(df['stage'] == l) & (df['method'].isna())][k].to_numpy()
    manual = df[(df['stage'] == l) & (df['method'] == 'Manual')][k].to_numpy()
    hands = df[(df['stage'] == l) & (df['method'] == 'Hands')][k].to_numpy()
    idle = df[(df['stage'] == l) & (df['method'] == 'Idle')][k].to_numpy()
    target = df[(df['stage'] == l) & (df['method'] == 'Target')][k].to_numpy()

    compare_list = []
    compares = []
    if (len(none) != 0):
      compares.append(none)
      compare_list.append("none")
    if (len(manual) != 0):
      compares.append(manual)
      compare_list.append("manual")
    if (len(hands) != 0):
      compares.append(hands)
      compare_list.append("hands")
    if (len(idle) != 0):
      compares.append(idle)
      compare_list.append("idle")
    if (len(target) != 0):
      compares.append(target)
      compare_list.append("target")

    print("\n[Comparing]\n0:", compare_list[0], "\n1:", compare_list[1], "\n2:", compare_list[2])
    print("\n++ Shapiro ++")
    p_shap = [0, 0, 0]
    for f, g in enumerate(compares):
      p_shap[f] = scipy.stats.shapiro(g)[1]
    print("Shapiro: ", p_shap)
    # p_eqv = scipy.stats.levene(compares[0], compares[1], compares[2])[1]

    if all(p > 0.05 for p in p_shap):
      print("⫸ Parametric")

      # print("++ one-way ANOVA ++")
      # stat = scipy.stats.f_oneway(compares[0], compares[1], compares[2])
      # p_value = stat.pvalue
      # print(stat)

      print("++ repeated measurements one-way ANOVA ++")
      df_anv = df[['user', 'method', 'stage', k]].copy()
      df_anv = df_anv.fillna("None")
      df_anv = df_anv[df_anv['stage'] == l]
      df_anv = df_anv.drop('stage', axis=1)
      df_anv.columns = ['Subject', 'Condition', 'Measurement']
      # display(df_anv)

      # obs_counts = df_anv.groupby(['Subject', 'Condition']).size()
      # print(obs_counts)

      print("++ Mauchly Spherical Test ++")
      mauchly_test = pg.sphericity(data=df_anv, dv='Measurement', within='Condition', subject='Subject')
      print("Mauchly Test:", mauchly_test)
      if (mauchly_test[4] > 0.05):
        print("⏺️ Sphericity passed")
        anova_rm = AnovaRM(df_anv, 'Measurement', 'Subject', within=['Condition'])
        result = anova_rm.fit()
        # print(result.summary())
        anova_table = result.anova_table
        # print(anova_table)

        F_statistic = anova_table['F Value'][0]
        p_value = anova_table['Pr > F'][0]
        print(f"F-statistic: {F_statistic}, P-value: {p_value}")
      else:
        print("⛔️ Sphericity failed")
        aov = pg.rm_anova(data=df_anv, dv='Measurement', within='Condition', subject='Subject', detailed=True, correction=True)
        print(aov)

      if (p_value < 0.05):
        print('🌈', compare_list[0], compare_list[1], compare_list[2], "have significant differences with p-value", p_value)
        print("++ Tukey's HSD ++")
        stat = scipy.stats.tukey_hsd(compares[0], compares[1], compares[2])
        print(stat)
        for q, r in enumerate(stat.pvalue):
          for s, t in enumerate(r):
            if (t < 0.05):
              print('🔅', compare_list[q], "and", compare_list[s], "have significant differences with p-value", t)
    else:
      print("⫷ Non-parametric")
      print("++ Kruskal-Wallis ++")
      stat = scipy.stats.kruskal(compares[0], compares[1], compares[2])
      print(stat)
      if (stat.pvalue < 0.05):
        print('🌈', compare_list[0], compare_list[1], compare_list[2], "have significant differences with p-value", stat.pvalue)
        print("++ Dunn's with Bonferroni ++")
        p_values = sp.posthoc_dunn(compares, p_adjust='bonferroni')
        print(p_values)

# Deprecated

In [ ]:
# overall methods comparing (with all stages together)
for i, k in enumerate(df.columns):
  if (i < 3): continue
  print("\n\n🔷Column", k)
  print("\n🔹Stage", l)
  none = df[df['method'].isna()][k].to_numpy()
  manual = df[df['method'] == 'Manual'][k].to_numpy()
  hands = df[df['method'] == 'Hands'][k].to_numpy()
  idle = df[df['method'] == 'Idle'][k].to_numpy()
  target = df[df['method'] == 'Target'][k].to_numpy()

  compare_list = ['None', 'Manual', 'Hands', 'Idle', 'Target']

  print("\n[Comparing]\n0:", compare_list[0], "\n1:", compare_list[1], "\n2:", compare_list[2], "\n3:", compare_list[3], "\n4:", compare_list[4])
  stat = scipy.stats.tukey_hsd(none, manual, hands, idle, target)
  print(stat)
  for q, r in enumerate(stat.pvalue):
    for s, t in enumerate(r):
      if (t < 0.05):
        print(compare_list[q], "and", compare_list[s], "have significant differences with p-value", t)
        

# Divided Comparison (SigGroup)

In [ ]:
# t-test / wilcoxon for specific items (divided by stages)

com_set = [
  ['head_qu_distance', '1 puzzle', 'None', 'Hands'],
  ['head_qu_mean_velocity', '1 puzzle', 'None', 'Hands'],
  ['player_mean_acc', '1 puzzle', 'None', 'Hands'],
  ['pref', '2 shooting', 'None', 'Manual']
]

for i, k in enumerate(com_set):
  print("///", k[0], "in task", k[1], "comparing methods of", k[2], "and", k[3] ,"///")

  if (k[2] == 'None'):
    a = df[(df['method'].isna()) & (df['stage'] == k[1])][k[0]]
  else:
    a = df[(df['method'] == k[2]) & (df['stage'] == k[1])][k[0]]
  if (k[3] == 'None'):
    b = df[(df['method'].isna()) & (df['stage'] == k[1])][k[0]]
  else:
    b = df[(df['method'] == k[3]) & (df['stage'] == k[1])][k[0]]

  print('++ Shapiro-Wilk Normality Test ++')
  stat1, p1 = scipy.stats.shapiro(a)
  print('Statistics=%.4f, p=%.4f' % (stat1, p1))
  stat2, p2 = scipy.stats.shapiro(b)
  print('Statistics=%.4f, p=%.4f' % (stat2, p2))

  print('++ Levene Test for Equal Variance ++')
  stat_eqv, p_eqv = scipy.stats.levene(a, b)
  print('Statistics=%.4f, p=%.4f' % (stat_eqv, p_eqv))
  
  if (p1 > 0.05 and p2 > 0.05):
    print('\n⫸ Parametric')
    print('\n- paired -')
    stat, p= scipy.stats.ttest_rel(
                                a = a, 
                                b = b,
                                alternative='greater'
                              )
    print("t-test: ", stat, " | p-value", p)
    if (p < 0.05 / 3):
      print('✅', k[2], 'is greater than', k[3], 'in terms of', k[0], 'in stage', k[1])
    else:
      print(k[2], 'is not greater than', k[3], 'in terms of', k[0], 'in stage', k[1])
    print("-◎-")
    stat, p= scipy.stats.ttest_rel(
                                a = b, 
                                b = a,
                                alternative='greater'
                              )
    print("t-test: ", stat, " | p-value", p)
    if (p < 0.05 / 3):
      print('✅', k[3], 'is greater than', k[2], 'in terms of', k[0])
    else:
      print(k[3], 'is not greater than', k[2], 'in terms of', k[0])
    print("\n--◎--◎--◎--")
    stat, p= scipy.stats.ttest_rel(
                                a = a, 
                                b = b,
                                alternative='less'
                              )
    print("t-test: ", stat, " | p-value", p)
    if (p < 0.05 / 3):
      print('✅', k[2], 'is less than', k[3], 'in terms of', k[0], 'in stage', k[1])
    else:
      print(k[2], 'is not less than', k[3], 'in terms of', k[0], 'in stage', k[1])
    print("-◎-")
    stat, p= scipy.stats.ttest_rel(
                                a = b, 
                                b = a,
                                alternative='less'
                              )
    print("t-test: ", stat, " | p-value", p)
    if (p < 0.05 / 3):
      print('✅', k[3], 'is less than', k[2], 'in terms of', k[0])
    else:
      print(k[3], 'is not less than', k[2], 'in terms of', k[0])
    print("\n--◎--◎--◎--")
    stat, p= scipy.stats.ttest_rel(
                                a = a, 
                                b = b,
                                alternative='two-sided'
                              )
    print("t-test: ", stat, " | p-value", p)
    if (p < 0.05 / 3):
      print('✅', k[2], 'is different to', k[3], 'in terms of', k[0], 'in stage', k[1])
    else:
      print(k[2], 'is not different to', k[3], 'in terms of', k[0], 'in stage', k[1])
  else:
    print('\n⫷ Non-parametric')
    print('\n- paired -')
    stat, p = scipy.stats.wilcoxon(
                                x = a, 
                                y = b, 
                                alternative='greater'
                              )
    print("Wilcoxon signed rank sum test: ", stat, " | p-value", p)
    if (p < 0.05 / 3):
      print('✅', k[2], 'is greater than', k[3], 'in terms of', k[0], 'in stage', k[1])
    else:
      print(k[2], 'is not greater than', k[3], 'in terms of', k[0], 'in stage', k[1])
    print("-◎-")
    stat, p= scipy.stats.wilcoxon(
                                x = b, 
                                y = a, 
                                alternative='greater'
                              )
    print("Wilcoxon signed rank sum test: ", stat, " | p-value", p)
    if (p < 0.05 / 3):
      print('✅', k[3], 'is greater than', k[2], 'in terms of', k[0], 'in stage', k[1])
    else:
      print(k[3], 'is not greater than', k[2], 'in terms of', k[0], 'in stage', k[1])
    print("\n--◎--◎--◎--")
    stat, p = scipy.stats.wilcoxon(
                                x = a, 
                                y = b, 
                                alternative='less'
                              )
    print("Wilcoxon signed rank sum test: ", stat, " | p-value", p)
    if (p < 0.05 / 3):
      print('✅', k[2], 'is less than', k[3], 'in terms of', k[0], 'in stage', k[1])
    else:
      print(k[2], 'is not less than', k[3], 'in terms of', k[0], 'in stage', k[1])
    print("-◎-")
    stat, p= scipy.stats.wilcoxon(
                                x = b, 
                                y = a, 
                                alternative='less'
                              )
    print("Wilcoxon signed rank sum test: ", stat, " | p-value", p)
    if (p < 0.05 / 3):
      print('✅', k[3], 'is less than', k[2], 'in terms of', k[0], 'in stage', k[1])
    else:
      print(k[3], 'is not less than', k[2], 'in terms of', k[0], 'in stage', k[1])
    print("\n--◎--◎--◎--")
    stat, p = scipy.stats.wilcoxon(
                                x = a, 
                                y = b, 
                                alternative='two-sided'
                              )
    print("Wilcoxon signed rank sum test: ", stat, " | p-value", p)
    if (p < 0.05 / 3):
      print('✅', k[2], 'is different to', k[3], 'in terms of', k[0], 'in stage', k[1])
    else:
      print(k[2], 'is not different to', k[3], 'in terms of', k[0], 'in stage', k[1])
  print("\n\n")

In [ ]:
# # non-parametric methods for specific items (divided by stages)

# com_set = [
#   ['head_qu_distance', '1 puzzle', 'None', 'Hands'],
#   ['head_qu_mean_velocity', '1 puzzle', 'None', 'Hands'],
#   ['player_mean_acc', '1 puzzle', 'None', 'Hands'],
#   ['pref', '2 shooting', 'None', 'Manual']
# ]

# for i, k in enumerate(com_set):
#   print("///", k[0] ,"///")
  
#   if (k[2] == 'None'):
#     a = df[(df['method'].isna()) & (df['stage'] == k[1])][k[0]]
#   else:
#     a = df[(df['method'] == k[2]) & (df['stage'] == k[1])][k[0]]
#   if (k[3] == 'None'):
#     b = df[(df['method'].isna()) & (df['stage'] == k[1])][k[0]]
#   else:
#     b = df[(df['method'] == k[3]) & (df['stage'] == k[1])][k[0]]

#   stat, p = scipy.stats.kruskal(a.to_numpy(), b.to_numpy())
#   print("\nKruskal-Wallis test: ", stat, " | p-value", p)
#   if (p < 0.05):
#     print('✅', k[2], 'is not the same as', k[3], 'in terms of', k[0], 'in stage', k[1])
#   else:
#     print(k[2], 'is the same as', k[3], 'in terms of', k[0], 'in stage', k[1])

#   stat3, p3 = scipy.stats.mannwhitneyu(a.to_numpy(), b.to_numpy(), alternative='greater')
#   print("\nMann Whitney U test: ", stat3, " | p-value", p3)
#   if (p3 < 0.05):
#     print('✅', k[2], 'is greater than', k[3], 'in terms of', k[0], 'in stage', k[1])
#   else:
#     print(k[2], 'is not greater than', k[3], 'in terms of', k[0], 'in stage', k[1])
#   print("--")
#   stat3, p3 = scipy.stats.mannwhitneyu(b.to_numpy(), a.to_numpy(), alternative='greater')
#   print("Mann Whitney U test: ", stat3, " | p-value", p3)
#   if (p3 < 0.05):
#     print('✅', k[3], 'is greater than', k[2], 'in terms of', k[0])
#   else:
#     print(k[3], 'is not greater than', k[2], 'in terms of', k[0])
  
#   print("\n\n")

# All Pairwise Comparison (Greedy)

## ❌ Need to revise ❌ 

In [ ]:
# t-test / wilcoxon for specific items (greedy)

com_set = [
  ['head_qu_distance', 'None', 'Idle'],
  ['head_qu_distance', 'Manual', 'Idle'],
  ['head_qu_distance', 'Target', 'Idle'],
  ['head_qu_mean_velocity', 'None', 'Idle'],
  ['head_qu_mean_velocity', 'Manual', 'Idle'],
  ['head_qu_mean_velocity', 'Target', 'Idle'],
  ['rightHand_qu_distance', 'Manual', 'Hands'],
  ['rightHand_qu_distance', 'Manual', 'Target'],
  ['rightHand_qu_distance', 'Hands', 'Idle'],
  ['rightHand_qu_distance', 'Target', 'Idle'],
  ['rightHand_qu_mean_velocity', 'Manual', 'Hands'],
  ['rightHand_qu_mean_velocity', 'Manual', 'Target'],
  ['rightHand_qu_mean_velocity', 'Hands', 'Idle'],
  ['rightHand_qu_mean_velocity', 'Target', 'Idle'],
  ['leftHand_qu_distance', 'None', 'Idle'],
  ['leftHand_qu_distance', 'Target', 'Idle'],
  ['leftHand_qu_distance', 'Hands', 'Idle'],
  ['leftHand_qu_mean_velocity', 'None', 'Idle'],
  ['leftHand_qu_mean_velocity', 'Target', 'Idle'],
  ['leftHand_qu_mean_acc', 'None', 'Idle'],
  ['player_mean_velocity', 'None', 'Idle'],
  ['player_mean_velocity', 'Manual', 'Idle'],
  ['player_mean_velocity', 'Hands', 'Idle'],
  ['player_mean_velocity', 'Target', 'Idle'],
  ['player_mean_velocity', 'None', 'Hands'],
  ['player_mean_acc', 'None', 'Idle'],
  ['player_mean_acc', 'Manual', 'Idle'],
  ['player_mean_acc', 'Hands', 'Idle'],
  ['player_mean_acc', 'Target', 'Idle'],
  ['pref', 'None', 'Manual']
]

for i, k in enumerate(com_set):
  print("///", k[0] ,"///")
  
  if (k[1] == 'None'):
    a = df[df['method'].isna()][k[0]]
  else:
    a = df[df['method'] == k[1]][k[0]]
  if (k[2] == 'None'):
    b = df[df['method'].isna()][k[0]]
  else:
    b = df[df['method'] == k[2]][k[0]]

  print(k, 'Shapiro-Wilk Normality Test')
  stat1, p1 = scipy.stats.shapiro(a)
  print('Statistics=%.4f, p=%.4f' % (stat1, p1))
  stat2, p2 = scipy.stats.shapiro(b)
  print('Statistics=%.4f, p=%.4f' % (stat2, p2))

  print('Levene Test for Equal Variance')
  stat_eqv, p_eqv = scipy.stats.levene(a, b)
  print('Statistics=%.4f, p=%.4f' % (stat_eqv, p_eqv))
  
  if (p1 > 0.05 and p2 > 0.05 and p_eqv > 0.05):
    print('\n⫸ Parametric')
    print('\n- paired -')
    if (a.shape[0] == b.shape[0]):
      stat, p= scipy.stats.ttest_rel(
                                  a = a, 
                                  b = b,
                                  alternative='greater'
                                )
      print("\nt-test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[1], 'is greater than', k[2], 'in terms of', k[0])
      else:
        print(k[1], 'is not greater than', k[2], 'in terms of', k[0])
      print("-◎-")
      stat, p= scipy.stats.ttest_rel(
                                  a = b, 
                                  b = a,
                                  alternative='greater'
                                )
      print("t-test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[2], 'is greater than', k[1], 'in terms of', k[0])
      else:
        print(k[2], 'is not greater than', k[1], 'in terms of', k[0])
      print('\n- independent -')
      stat, p= scipy.stats.ttest_ind(
                                  a = a, 
                                  b = b,
                                  alternative='greater'
                                )
      print("\nt-test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[1], 'is greater than', k[2], 'in terms of', k[0])
      else:
        print(k[1], 'is not greater than', k[2], 'in terms of', k[0])
      print("-◎-")
      stat, p= scipy.stats.ttest_ind(
                                  a = b, 
                                  b = a,
                                  alternative='greater'
                                )
      print("t-test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[2], 'is greater than', k[1], 'in terms of', k[0])
      else:
        print(k[2], 'is not greater than', k[1], 'in terms of', k[0])
    else:
      print("🚫The size of samples are not equal")
  else:
    print('\n⫷ Non-parametric')
    print('\n- paired -')
    if (a.shape[0] == b.shape[0]):
      stat, p= scipy.stats.wilcoxon(
                                  x = a, 
                                  y = b, 
                                  alternative='greater'
                                )
      print("Wilcoxon signed rank sum test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[1], 'is greater than', k[2], 'in terms of', k[0])
      else:
        print(k[1], 'is not greater than', k[2], 'in terms of', k[0])
      print("-◎-")
      stat, p= scipy.stats.wilcoxon(
                                  x = b, 
                                  y = a, 
                                  alternative='greater'
                                )
      print("Wilcoxon signed rank sum test: ", stat, " | p-value", p)
      if (p < 0.05 / 3):
        print('✅', k[2], 'is greater than', k[1], 'in terms of', k[0])
      else:
        print(k[2], 'is not greater than', k[1], 'in terms of', k[0])
      print('\n- independent -')
      stat3, p3 = scipy.stats.mannwhitneyu(a.to_numpy(), b.to_numpy(), alternative='greater')
      print("Mann Whitney U test: ", stat3, " | p-value", p3)
      if (p3 < 0.05 / 3):
        print('✅', k[1], 'is greater than', k[2], 'in terms of', k[1], 'in terms of', k[0])
      else:
        print(k[1], 'is not greater than', k[2], 'in terms of', k[0])
      stat3, p3 = scipy.stats.mannwhitneyu(b.to_numpy(), a.to_numpy(), alternative='greater')
      print("Mann Whitney U test: ", stat3, " | p-value", p3)
      if (p3 < 0.05 / 3):
        print('✅', k[2], 'is greater than', k[1], 'in terms of', k[0])
      else:
        print(k[2], 'is not greater than', k[1], 'in terms of', k[0])
    else:
      print("🚫The size of samples are not equal")
  print("\n\n")

# Deprecated

In [ ]:
# # non-parametric methods for specific items (overall)

# com_set = [
#   ['head_qu_distance', 'None', 'Idle'],
#   ['head_qu_distance', 'Manual', 'Idle'],
#   ['head_qu_distance', 'Target', 'Idle'],
#   ['head_qu_mean_velocity', 'None', 'Idle'],
#   ['head_qu_mean_velocity', 'Manual', 'Idle'],
#   ['head_qu_mean_velocity', 'Target', 'Idle'],
#   ['rightHand_qu_distance', 'Manual', 'Hands'],
#   ['rightHand_qu_distance', 'Manual', 'Target'],
#   ['rightHand_qu_distance', 'Hands', 'Idle'],
#   ['rightHand_qu_distance', 'Target', 'Idle'],
#   ['rightHand_qu_mean_velocity', 'Manual', 'Hands'],
#   ['rightHand_qu_mean_velocity', 'Manual', 'Target'],
#   ['rightHand_qu_mean_velocity', 'Hands', 'Idle'],
#   ['rightHand_qu_mean_velocity', 'Target', 'Idle'],
#   ['leftHand_qu_distance', 'None', 'Idle'],
#   ['leftHand_qu_distance', 'Target', 'Idle'],
#   ['leftHand_qu_distance', 'Hands', 'Idle'],
#   ['leftHand_qu_mean_velocity', 'None', 'Idle'],
#   ['leftHand_qu_mean_velocity', 'Target', 'Idle'],
#   ['leftHand_qu_mean_acc', 'None', 'Idle'],
#   ['player_mean_velocity', 'None', 'Idle'],
#   ['player_mean_velocity', 'Manual', 'Idle'],
#   ['player_mean_velocity', 'Hands', 'Idle'],
#   ['player_mean_velocity', 'Target', 'Idle'],
#   ['player_mean_velocity', 'None', 'Hands'],
#   ['player_mean_acc', 'None', 'Idle'],
#   ['player_mean_acc', 'Manual', 'Idle'],
#   ['player_mean_acc', 'Hands', 'Idle'],
#   ['player_mean_acc', 'Target', 'Idle'],
#   ['pref', 'None', 'Manual']
# ]

# for i, k in enumerate(com_set):
#   print("///", k[0] ,"///")
  
#   if (k[1] == 'None'):
#     a = df[df['method'].isna()][k[0]]
#   else:
#     a = df[df['method'] == k[1]][k[0]]
#   if (k[2] == 'None'):
#     b = df[df['method'].isna()][k[0]]
#   else:
#     b = df[df['method'] == k[2]][k[0]]

#   stat, p = scipy.stats.kruskal(a.to_numpy(), b.to_numpy())
#   print("\nKruskal-Wallis test: ", stat, " | p-value", p)
#   if (p < 0.05 / 3):
#     print('✅', k[1], 'is not the same as', k[2], 'in terms of', k[0])
#   else:
#     print(k[1], 'is the same as', k[2], 'in terms of', k[0])

#   stat3, p3 = scipy.stats.mannwhitneyu(a.to_numpy(), b.to_numpy(), alternative='greater')
#   print("\nMann Whitney U test: ", stat3, " | p-value", p3)
#   if (p3 < 0.05 / 3):
#     print('✅', k[1], 'is greater than', k[2], 'in terms of', k[0])
#   else:
#     print(k[1], 'is not greater than', k[2], 'in terms of', k[0])
#   print("--")
#   stat3, p3 = scipy.stats.mannwhitneyu(b.to_numpy(), a.to_numpy(), alternative='greater')
#   print("Mann Whitney U test: ", stat3, " | p-value", p3)
#   if (p3 < 0.05 / 3):
#     print('✅', k[1], 'is greater than', k[2], 'in terms of', k[0])
#   else:
#     print(k[1], 'is not greater than', k[2], 'in terms of', k[0])
  
#   print("\n\n")

# Plot

In [ ]:
# plot about head_qu_distance ✅
_ = plt.boxplot([df[(df['method'] == 'Hands') & (df['stage'] == '1 puzzle')]['head_qu_distance'], df[(df['method'].isna()) & (df['stage'] == '1 puzzle')]['head_qu_distance']])
plt.xticks([1, 2], ['Hands', 'None'])
plt.ylabel('Head Quaternion Distance')
# title
plt.title('Head Quaternion Distance in Puzzle Stage in Hands Method and None')

In [ ]:
# plot about preference ✅
_ = plt.boxplot([df[(df['method'].isna()) & (df['stage'] == '2 shooting')]['pref'],
                 df[(df['method'] == 'Manual') & (df['stage'] == '2 shooting')]['pref'],
                 df[(df['method'] == 'Target') & (df['stage'] == '2 shooting')]['pref'], 
                 ])
plt.xticks([1, 2, 3], ['None', 'Manual', 'Target'])
plt.ylabel('Preference')
# title
plt.title('Preference in Shooting Stage in 2 Methods and None')

In [ ]:
# plot all columns methods comparisons with all stages aggregated
for i, col in enumerate(df.columns[7:]):
  fig = plt.boxplot([ df[df['method'].isna()][col],
                    df[df['method'] == 'Manual'][col],
                    df[df['method'] == 'Hands'][col], 
                    df[df['method'] == 'Idle'][col], 
                    df[df['method'] == 'Target'][col], 
                ])
  plt.xticks([1, 2, 3, 4, 5], ['Default', 'Manual', 'Arm', 'Locomotion', 'Target'])
  plt.ylabel(col)
  # title
  plt.title(col + ' in All Methods in all stages')
  plt.show()

In [ ]:
# plot all columns methods comparisons with each stage
for s, stage in enumerate(stage_list):
  for i, col in enumerate(df.columns[7:]):
    fig = plt.boxplot([ df[(df['method'].isna()) & (df['stage'] == stage)][col],
                      df[(df['method'] == 'Manual') & (df['stage'] == stage)][col],
                      df[(df['method'] == 'Hands') & (df['stage'] == stage)][col], 
                      df[(df['method'] == 'Idle') & (df['stage'] == stage)][col], 
                      df[(df['method'] == 'Target') & (df['stage'] == stage)][col], 
                  ])
    plt.xticks([1, 2, 3, 4, 5], ['Default', 'Manual', 'Arm', 'Locomotion', 'Target'])
    plt.ylabel(col)
    # title
    plt.title(col + ' in All Methods in ' + stage.split(' ')[1] + ' stage')
    plt.show()

In [ ]:
puzzle_arm_mean = df[(df['method'] == 'Hands') & (df['stage'] == '1 puzzle')]['head_qu_distance'].mean()
puzzle_none_mean = df[(df['method'].isna()) & (df['stage'] == '1 puzzle')]['head_qu_distance'].mean()
diff = 1 - df[(df['method'] == 'Hands') & (df['stage'] == '1 puzzle')]['head_qu_distance'].mean() / df[(df['method'].isna()) & (df['stage'] == '1 puzzle')]['head_qu_distance'].mean()

print("Puzzle Arm Mean: ", puzzle_arm_mean)
print("Puzzle None Mean: ", puzzle_none_mean)
print("Difference: ", diff)

In [ ]:
df[(df['method'] == 'Manual') & (df['stage'] == '2 shooting')]['pref'].mean()
df[(df['method'].isna()) & (df['stage'] == '2 shooting')]['pref'].mean()
1 - df[(df['method'] == 'Manual') & (df['stage'] == '2 shooting')]['pref'].mean() / df[(df['method'].isna()) & (df['stage'] == '2 shooting')]['pref'].mean()